In [1]:
# imports and data constructs

import numpy as np

bg = np.load("../data/constructs/basin_mushgraph.npz", allow_pickle=True) # basin_mushgraph.py
sl = np.load("../data/constructs/slope_torch.npz", allow_pickle=True)     # sledGraph.ipynb
pl = np.load("../data/constructs/pit_basin_links.npz", allow_pickle=True) # sledGraph.ipynb

N_s, N_b = sl["X"].shape[0], bg["X"].shape[0] # number slope domain, basin domain

In [2]:
# additional: connect each tree node to its corresponding pit

In [3]:
# import threshold trees to map nodes to their pits
import pickle
basinPit_trees = pickle.load(open("../data/constructs/sledgraph_torch_treehandoff.pkl","rb"))["basinPit_trees"]

In [4]:
import numpy as np

# build node-to-pit edges indices

node2idx, node_pit_key = {}, {}
for pit_key, tree in basinPit_trees.items():        # same order as the flatten pass
    for n in tree.nodes():
        if n not in node2idx:
            node2idx[n] = len(node2idx)
            node_pit_key[n] = pit_key               # every node's pit = this tree's root

assert len(node2idx) == N_s, (len(node2idx), N_s)

node_pit_idx = np.empty(N_s, dtype=np.int64)
for n, i in node2idx.items():
    node_pit_idx[i] = node2idx[node_pit_key[n]]     # torch idx of this node's pit
np.save("node_pit_idx.npy", node_pit_idx)

# guard: every pit must map to itself, and must agree with the saved pit_torch_idx
pit_t = sl["pit_torch_idx"]
assert (node_pit_idx[pit_t] == pit_t).all(), "pit self-map broken — ordering mismatch"
print("node→pit map OK | non-pit nodes:", int((np.arange(N_s) != node_pit_idx).sum()))

node→pit map OK | non-pit nodes: 876404


In [5]:
import torch
# set node-pit edges to torch edges
npi   = torch.tensor(node_pit_idx, dtype=torch.long)
sids  = torch.arange(N_s, dtype=torch.long)
mask  = sids != npi                                  # drop pit self-loops
# torch edge list
e_pitskip = torch.stack([sids[mask], npi[mask]])     # node -> its pit (876,404 edges)

In [6]:
# retrieve built edges and normalize

def zscore(X):                       # nan-robust: ignore NaN for stats, then NaN->0 (col mean)
    m = torch.nanmean(X, 0)
    v = torch.nanmean((X - m)**2, 0)
    return torch.nan_to_num((X - m) / torch.sqrt(v + 1e-6))


X_s = zscore(torch.tensor(sl["X"], dtype=torch.float32))
X_b = zscore(torch.tensor(bg["X"], dtype=torch.float32))
y_s = torch.tensor(sl["y"], dtype=torch.float32)

e_slope = torch.tensor(sl["edge_index"], dtype=torch.long)
e_basin = torch.tensor(bg["edge_index"], dtype=torch.long) + N_s # bumping past slope indices
pit_t   = torch.tensor(sl["pit_torch_idx"], dtype=torch.long)
e_cross = torch.stack([pit_t[torch.tensor(pl["link_pit"])],
                       torch.tensor(pl["link_basin"]) + N_s])

In [7]:
# finalize edge index for training
from torch_geometric.utils import to_undirected

#edge_index = to_undirected(torch.cat([e_slope, e_basin, e_cross], 1),
 #                          num_nodes=N_s + N_b) # bidirectional, dedup

edge_index = to_undirected(
    torch.cat([e_slope, e_basin, e_cross, e_pitskip], dim=1),
    num_nodes=N_s + N_b)
print(f"nodes {N_s+N_b:,} (slope {N_s:,} + basin {N_b:,}) | edges {edge_index.size(1):,} "
      f"| valid y {int(torch.isfinite(y_s).sum()):,}")

nodes 976,613 (slope 876,690 + basin 99,923) | edges 4,524,268 | valid y 876,690


In [8]:
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import TransformerConv

torch.manual_seed(0)
dev = "cuda" if torch.cuda.is_available() else "cpu"

ABLATE_BASIN = True    # <-- flip to True for the matched no-basin control
Xb_in = X_b[torch.randperm(N_b)] if ABLATE_BASIN else X_b   # shuffle kills spatial basin signal, keeps structure

valid = torch.where(torch.isfinite(y_s))[0]              # only slope nodes with a target
perm  = valid[torch.randperm(valid.numel())]
n_tr, n_va = int(0.70*perm.numel()), int(0.15*perm.numel())
tr, va, te = perm[:n_tr], perm[n_tr:n_tr+n_va], perm[n_tr+n_va:]

class HeteroGT(nn.Module):
    def __init__(self, in_s, in_b, hid=64, heads=4, layers=4):
        super().__init__()
        self.enc_s = nn.Linear(in_s, hid); self.enc_b = nn.Linear(in_b, hid)
        self.convs = nn.ModuleList(
            [TransformerConv(hid, hid // heads, heads=heads, concat=True) for _ in range(layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hid) for _ in range(layers)])
        self.head  = nn.Linear(hid, 1)
    def forward(self, xs, xb, ei):
        h = torch.cat([self.enc_s(xs), self.enc_b(xb)], 0)
        for conv, norm in zip(self.convs, self.norms):
            h = norm(h + F.relu(conv(h, ei)))     # residual + LN transformer block
        return self.head(h).squeeze(-1)

model = HeteroGT(X_s.size(1), X_b.size(1)).to(dev)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
Xs, Xb, ei, y = X_s.to(dev), Xb_in.to(dev), edge_index.to(dev), y_s.to(dev)
tr, va, te = tr.to(dev), va.to(dev), te.to(dev)

def r2(p, idx):
    t = y[idx]; return (1 - ((p[idx]-t)**2).sum() / ((t-t.mean())**2).sum()).item()

best = -1e9
for ep in range(1, 1001):
    model.train(); opt.zero_grad()
    out = model(Xs, Xb, ei); F.mse_loss(out[tr], y[tr]).backward(); opt.step()
    if ep % 25 == 0:
        model.eval()
        with torch.no_grad():
            out = model(Xs, Xb, ei); vr, tr2 = r2(out, va), r2(out, te)
        best = max(best, vr)
        print(f"ep {ep:4d} | loss {F.mse_loss(out[tr], y[tr]):.4f} | val R² {vr:.3f} | test R² {tr2:.3f}")
print(f"done ({'NO-basin' if ABLATE_BASIN else 'basin'}) | best val R² {best:.3f}")

ep   25 | loss 8.2683 | val R² 0.311 | test R² 0.309
ep   50 | loss 6.8112 | val R² 0.433 | test R² 0.430
ep   75 | loss 5.7519 | val R² 0.519 | test R² 0.518
ep  100 | loss 5.1240 | val R² 0.572 | test R² 0.569
ep  125 | loss 4.6479 | val R² 0.611 | test R² 0.608
ep  150 | loss 4.4756 | val R² 0.626 | test R² 0.621
ep  175 | loss 4.1716 | val R² 0.651 | test R² 0.647
ep  200 | loss 4.0256 | val R² 0.663 | test R² 0.659
ep  225 | loss 3.9205 | val R² 0.671 | test R² 0.668
ep  250 | loss 3.7388 | val R² 0.686 | test R² 0.682
ep  275 | loss 3.7316 | val R² 0.686 | test R² 0.681
ep  300 | loss 3.5478 | val R² 0.701 | test R² 0.697
ep  325 | loss 3.5516 | val R² 0.700 | test R² 0.696
ep  350 | loss 3.3810 | val R² 0.714 | test R² 0.710
ep  375 | loss 3.3385 | val R² 0.717 | test R² 0.714
ep  400 | loss 3.2964 | val R² 0.720 | test R² 0.717
ep  425 | loss 3.2016 | val R² 0.727 | test R² 0.724
ep  450 | loss 3.1795 | val R² 0.729 | test R² 0.726
ep  475 | loss 3.1050 | val R² 0.735 | test R²